In [10]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage,HumanMessage
import operator
from typing import TypedDict,Annotated
from langgraph.checkpoint.memory import MemorySaver
load_dotenv()

True

In [11]:
model=ChatGroq(
    model="llama-3.3-70b-versatile"
)

In [12]:
class message_state(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [13]:
def chat_llm(state:message_state):
    message=state['messages']
    response=model.invoke(message)
    return {"messages":[response]}

In [14]:
check_point=MemorySaver()
graph=StateGraph(message_state)
graph.add_node("chat_llm",chat_llm)
graph.add_edge(START,"chat_llm")
graph.add_edge("chat_llm",END)

workflow=graph.compile(checkpointer=check_point)

In [16]:
# user_input="hello"
# output=workflow.invoke({"messages":user_input})
# output["messages"][0].content

In [17]:
thread_id="1"
while True:
    user_input=input("Type here :")
    print("user :",user_input)
    if user_input.strip().lower() in ["exit", 'bye', 'quit']:
        break
    config={"configurable":{"thread_id":thread_id}}
    output=workflow.invoke({"messages":user_input},config=config)
    response=output["messages"][-1].content
    print("AI :",response)

user : hello
AI : Hello. How can I assist you today?
user : im zain
AI : Hello Zain, it's nice to meet you. Is there something I can help you with or would you like to chat?
user : tell my name
AI : Your name is Zain.
user : how you know
AI : I know your name because you told me earlier. You said "im zain" when we started chatting.
user : ok bye see you
AI : It was nice chatting with you, Zain. See you later. Bye!
user : bye
